# Trabajo Final Integrador N°2

**IFTS N°24 - Técnicas de Procesamiento de Imágenes**

Profesores: Matías Barreto - Cynthia Villagra

Estudiante: Cintia Coronel

**Contexto:**

Trabajo habitualmente con la clasificación de materiales y sustancias en el aula de química: cómo se separan, qué propiedades los distinguen, y cómo eso determina su destino final. Esto invita a reflexionar sobre el reciclaje: ¿todos los materiales se pueden reciclar? ¿qué tipos de materiales existen y cómo se clasifican en la práctica, fuera del aula?

A partir de estas preguntas, busqué modelos en HF para estudiar cómo la IA puede ayudar a identificar el tipo de residuo a partir de una foto, y sugerir cómo manejarlo correctamente.

De hecho, recordé sobre este robot ecológico (https://youtu.be/A2ZCZsJj2OU?si=9OF6NchCn7g6wBkJ) que había visto hace unos años en los medios.

Por último, si bien esta práctica ayuda a probar la capacidad de un modelo para clasificar residuos y sus aplicaciones en el aula, siempre es recomendable invitar a la reflexión sobre cuánta basura generamos y qué hacemos con ella.

**Selección del modelo:**

Luego de revisar las opciones de Hugging Face, opté por watersplash/waste-classification, entrenado con el dataset Garbage Classification (https://www.kaggle.com/datasets/mostafaabla/garbage-classification), que tiene 12 clases y reúne 15.150 imágenes en total. Las fotos son reales (no ilustraciones ni dibujos), y aunque algunas clases tienen más imágenes que otras, todas superan las 600.

Además, el modelo reporta un accuracy del 98%, es decir que de 100 pruebas, acertó el 98 de los casos con ese dataset. Veamos qué obtenemos en nuestras pruebas.


## Trabajo con IA

Podes usar IA como apoyo para discutir alternativas, revisar errores de codigo o auditar una decision tecnica. Pero no vale usarla como reemplazo de tu criterio. Toda seleccion de parametros, toda justificacion y toda version entregada tienen que quedar bajo tu responsabilidad.

### Registro breve del uso de IA


| Caso | Objetivo de la consulta | Pedido a la IA | Que conservaste y por que | Que descartaste y por que | Que aprendiste |
|---|---|---|---|---|---|
| Traducción |Traducir las clases a español|Qué operación lógica conviene realizar para la traducción de las clases, en qué espacio del código es más óptimo realizarlo |Conservé la estructura lógica de traducción de la clase, con el armado del diccionario|Descarté los emojis| Si bien funcionaría si lo cambiamos a líneas más abajo, queda mejor ordenado en lectura si se coloca en la primera parte para que vaya en la sección de Data Layer *"esto es el modelo, y esto es cómo interpreto lo que devuelve"* / También me sirvió para estructurar las recomendaciones de qué hacer con cada categoría
| Recomendación educativa |Cómo mostrar la salida de la recomendación | En qué sector ubicar la salida, el código para que muestre la recomendación en base a la categoría de mayor probabilidad| Mantuve el código | -- | El orden de los bloques de cada sección | 


## Resolución

### 🖥️ *App.py*

In [5]:
%%writefile app.py
import os
import gradio as gr
from transformers import pipeline
 
print("Configurando las capas de la aplicación...")
 
# =====================================================================
# 1. DATA LAYER (Capa de Datos): Carga del modelo preentrenado
# =====================================================================
print("[Data Layer] Cargando modelo preentrenado...")
clasificador_modelo = pipeline(
    "image-classification",
    model="watersplash/waste-classification"
)
print("✓ [Data Layer] Modelo cargado con éxito en memoria.")
 
# El modelo cargado devuelve las clases de cada material en inglés. Las traducimos para mostrar la respuesta en castellano.
TRADUCCIONES = {
    "battery": "Pila/Batería",
    "biological": "Orgánico",
    "brown-glass": "Vidrio color ámbar",
    "cardboard": "Cartón",
    "clothes": "Ropa",
    "green-glass": "Vidrio verde",
    "metal": "Metal",
    "paper": "Papel",
    "plastic": "Plástico",
    "shoes": "Calzado",
    "trash": "Basura general",
    "white-glass": "Vidrio sin color",
}
 
# Recomendación práctica de manejo y/o reciclaje según la categoría detectada
RECOMENDACIONES = {
    "Pila/Batería": "⚠️ Es un residuo peligroso. No lo tires al tacho de basura común: llevalo a un punto de recolección especial.",
    "Orgánico": "🌱 Es ideal para el compost. Separalo por completo del resto de los residuos.",
    "Vidrio ámbar": "♻️ Enjuagá el envase y retirá la tapa metálica antes de llevarlo a reciclar.",
    "Cartón": "♻️ Desarmá las cajas y retirá los excedentes de cintas, ganchos metálicos, stickers o restos de plástico antes de reciclar.",
    "Ropa": "👕 Si está en buen estado, donala. También podés buscar un punto de reciclaje de textil.",
    "Vidrio verde": "♻️ Enjuagá el envase y retirá la tapa antes de llevarlo a reciclar.",
    "Metal": "♻️ Enjuagá latas de comida o bebida y aplastalas para ahorrar espacio. Las latas rígidas pueden reutilizarse.",
    "Papel": "♻️ Reciclalo si está limpio y seco. Evitá papel sucio, plastificado o con grasa.",
    "Plástico": "♻️ Lavá y secá el envase. Si podés, separalo por tipo de plástico (PET, HDPE, PVC, LDPE, PP, PS, otros).",
    "Calzado": "👟 Si está en buen estado, donalo. También podés buscar un punto de reciclaje de calzado/textil para dejarlo.",
    "Basura general": "🗑️ No es reciclable: va al cesto de residuos comunes.",
    "Vidrio blanco": "♻️ Enjuagá el envase y retirá la tapa antes de llevarlo a reciclar.",
}
 
# =====================================================================
# 2. BUSINESS LOGIC LAYER (Capa de Lógica de Negocio): Validación y control
# =====================================================================
def clasificar_residuo(imagen):
    """
    Valida los datos y ejecuta el flujo.
    """
    if imagen is None:
        print("✗ [Business Layer] Intento de clasificación sin imagen.")
        return {"Error": "Por favor, suban una imagen válida."}, "Subí una imagen para ver la recomendación."
 
    print("[Business Layer] Imagen recibida. Ejecutando preprocesamiento e inferencia...")
 
    try:
        # Inferencia directa mediante DataLayer
        resultados = clasificador_modelo(imagen)
 
        # Formateamos y traducimos el resultado al castellano
        salida = {
            TRADUCCIONES.get(r["label"].lower(), r["label"]): float(r["score"]) # sin el lower, no tendríamos recomendaciones
            for r in resultados
        }
 
        # Tomamos la categoría con mayor probabilidad para buscar su recomendación
        categoria_top = max(salida, key=salida.get)
        recomendacion = RECOMENDACIONES.get(
            categoria_top,
            "No hay una recomendación específica para esta categoría todavía."
        )
 
        print(f"✓ [Business Layer] Clasificación completada: {categoria_top}.")
        return salida, recomendacion
 
    except Exception as error:
        print(f"✗ [Business Layer] Falla en la inferencia: {error}")
        return {"Error de inferencia": str(error)}, "Ocurrió un error al generar la recomendación."
 
# =====================================================================
# 3. PRESENTATION LAYER (Capa de Presentación): Interfaz de Gradio
# =====================================================================
print("[Presentation Layer] Construyendo interfaz web...")
 
demo = gr.Interface(
    fn=clasificar_residuo,
    inputs=gr.Image(type="pil", label="Subí una foto del residuo"),
    outputs=[
        gr.Label(num_top_classes=5, label="Predicción"),
        gr.Textbox(label="Qué hacer con este residuo", lines=3),
    ],
    title="Clasificador de Residuos",
    description=(
        "Subí una imagen y el modelo va a predecir a qué categoría de residuo pertenece "
        "(Pila/Batería, Orgánico, Cartón, Vidrio, Metal, Papel, Plástico, Ropa, Calzado, Basura general) "
        "y te va a sugerir cómo manejarlo."
    ),
)
 
print("✓ [Presentation Layer] UI inicializada correctamente.")
 
# =====================================================================
# 4. LANZAMIENTO DE LA APLICACIÓN
# =====================================================================
if __name__ == "__main__":
    print("✦ Iniciando el servidor de Gradio...")
    demo.launch()

Writing app.py


### 🖥️ *Requirements.txt*

Las cuatro librerías que necesito son:
- gradio
- transformers
- torch
- pillow

In [3]:
%%writefile requirements.txt
transformers>=4.35.0
torch>=2.0.0
pillow>=9.0.0
gradio>=4.0.0

Writing requirements.txt


### 🖥️ *README.md*

Hugging Face lee este archivo para configurar el Space antes de levantarlo. 

In [6]:
%%writefile README.md
---
title: Clasificador de Residuos
emoji: ♻️
colorFrom: green
colorTo: blue
sdk: gradio
sdk_version: 4.44.1
app_file: app.py
pinned: false
license: mit
---

# Clasificador de Residuos

Aplicación web que clasifica imágenes de residuos en distintas categorías (plástico, papel, metal, vidrio, cartón, etc.) 
utilizando un modelo Vision Transformer (ViT) preentrenado, disponible en Hugging Face Hub: https://huggingface.co/watersplash/waste-classification.

## Cómo funciona

1. El usuario sube una foto de un residuo.
2. El modelo predice a qué categoría pertenece.
3. Se muestran las 5 categorías más probables junto con su nivel de confianza, y una recomendación de manejo para la categoría más probable.

## Arquitectura

La aplicación sigue un patrón de 3 capas:

- **Data Layer**: carga del modelo preentrenado (`pipeline` de `transformers`).
- **Business Logic Layer**: valida la imagen recibida, ejecuta la inferencia y formatea/traduce el resultado.
- **Presentation Layer**: interfaz interactiva construida con Gradio (`gr.Interface`).

Overwriting README.md


### Mejoras a futuro:

- Reacomodar para trabajar con los tipos de tachos que hay en las escuelas de la Ciudad
- Mapa interactivo con los puntos verdes de la Ciudad para que puedan dejar residuos específicos, además de locales donde puedan dejar textiles